# DAY 2 — From Ollama to LangChain: Chains, Memory & Document Loading

**Yesterday** we talked to Ollama directly using `requests` and prompt engineering.  
**Today** we introduce **LangChain** — a framework that gives us reusable building blocks:  
chains, memory, document loaders, text splitters, and more.

By the end of this notebook you will:
1. Connect to Ollama **through LangChain**
2. Build a **prompt + LLM chain** (the core LangChain pattern)
3. Add **conversational memory** so the LLM remembers previous messages
4. Load PDFs, inspect their **metadata**, and split them into **chunks**
5. **Filter chunks by metadata** — the foundation of selective retrieval

---

## Part 1 — Setup & Installations

We need a few packages. Run this cell once.

In [ ]:
!pip install langchain langchain-ollama langchain-community pypdf -q

### Imports

Here's what each import does:

| Import | Purpose |
|--------|---------|
| `ChatOllama` | LangChain wrapper around your local Ollama models |
| `ChatPromptTemplate` | Build reusable prompt templates with placeholders |
| `StrOutputParser` | Extracts the plain string from the LLM's response |
| `HumanMessage`, `AIMessage`, `SystemMessage` | Message types — the building blocks of a conversation |
| `PyPDFLoader` | Loads PDF files as LangChain Document objects |
| `RecursiveCharacterTextSplitter` | Splits long text into overlapping chunks |

In [3]:
# LLM
from langchain_ollama import ChatOllama

# Prompts & output parsing
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Message types — we'll use these to build conversation history
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Document loading & splitting
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Standard library
from datetime import date

print("All imports successful!")

All imports successful!


In [26]:
# Create the LLM object — make sure Ollama is running with llama3.2:3b pulled
llm = ChatOllama(
    model="qwen2.5:1.5b",
    temperature=0.7
)

# Quick test — just invoke it with a string
response = llm.invoke("What is LangChain in one sentence?")
print(response.content)

LangChain is a toolkit for building and deploying large language models on the chain of hardware and software.


> **What just happened?**  
> `llm.invoke()` sent your message to Ollama and returned an `AIMessage` object.  
> We access the text via `.content`. No JSON parsing, no headers — LangChain handled it.

---
## Part 3 — Building Your First Chain

A **chain** in LangChain connects components together using the pipe (`|`) operator:

```
Prompt Template  →  LLM  →  Output Parser
```

This is the most fundamental pattern in LangChain. Once you understand this, everything else is just adding more links to the chain.

### Why use a chain instead of calling `llm.invoke()` directly?
- **Reusable templates** — define the prompt once, swap variables
- **Composable** — plug in memory, retrievers, tools later
- **Clean** — separate prompt logic from LLM logic

In [27]:
# Step 1: Define a prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor. Explain concepts clearly and briefly."),
    ("human", "{question}")
])

# Step 2: Build the chain — Prompt → LLM → Parse to string
chain = prompt | llm | StrOutputParser()

# Step 3: Run the chain
answer = chain.invoke({"question": "What is a vector database? Explain in 2 sentences"})
print(answer)

A vector database stores and retrieves data in a way that each piece of information is represented as a vector, typically a point in a high-dimensional space. This allows for efficient similarity-based search and can be used for applications like image search, video indexing, and recommendation systems.


> **Breaking it down:**
> 1. `prompt` fills in `{question}` and creates the formatted messages
> 2. `llm` sends those messages to Ollama and gets back an AI response
> 3. `StrOutputParser()` extracts just the text string from the response
>
> The `|` pipe operator chains them: output of one becomes input to the next.

In [28]:
# Try it with a different question — same chain, different input
answer = chain.invoke({"question": "Give me 2 points clarifying why langchain is a framework as well as library. keep it short and precise"})
print(answer)

1. LangChain is a library that provides a unified framework for building and managing large language models, making it easier to create and deploy complex models.
2. It offers pre-defined components and interfaces that can be combined to create custom language models, streamlining the process and reducing the need for extensive coding.


---
## Part 4 — Adding Memory to the Chat

Right now, each `llm.invoke()` is **stateless** — the LLM has no idea what you asked before.  
To make it remember, we manage a **list of messages** ourselves.

### How it works:
1. We keep a **Python list** called `chat_history`
2. Before each call, we pass the **full list** to the LLM
3. After the LLM replies, we **append** both the user message and AI response to the list
4. Next time, the LLM sees everything that came before

That's it. No magic — memory is just **"send the old messages again"**.

### LangChain message types:
- `SystemMessage(...)` — sets the LLM's behavior (persona, rules)
- `HumanMessage(...)` — what the user said
- `AIMessage(...)` — what the LLM replied

In [29]:
# Start with a system message and an empty history
chat_history = [
    SystemMessage(content="You are a friendly AI assistant. Keep your replies short and helpful.")
]

# Helper function: send a message and update the history
def chat(user_input):
    chat_history.append(HumanMessage(content=user_input))
    response = llm.invoke(chat_history)
    chat_history.append(AIMessage(content=response.content))
    return response.content

In [30]:
# First message
print(chat("Hi! My name is Shiva."))

Hello Shiva! It's nice to meet you. How can I help you today?


In [31]:
# Second message — does it remember your name?
print(chat("What is my name?"))

Your name is Shiva.


In [32]:
# Third message — the history keeps growing
print(chat("What have we talked about so far? Only use the context in the message list"))

You've been chatting about friendly AI assistants and asking about your name.


In [33]:
# Peek inside the history — see exactly what the LLM receives
print(f"Total messages in history: {len(chat_history)}\n")
for msg in chat_history:
    role = type(msg).__name__.replace("Message", "")
    print(f"[{role}]: {msg.content[:100]}{'...' if len(msg.content) > 100 else ''}")
    print()

Total messages in history: 7

[System]: You are a friendly AI assistant. Keep your replies short and helpful.

[Human]: Hi! My name is Shiva.

[AI]: Hello Shiva! It's nice to meet you. How can I help you today?

[Human]: What is my name?

[AI]: Your name is Shiva.

[Human]: What have we talked about so far? Only use the context in the message list

[AI]: You've been chatting about friendly AI assistants and asking about your name.



> **Key Takeaway:**  
> There is no magic behind "memory" — it's just a list of messages that you send to the LLM every time.  
> The LLM itself is stateless. **You** are the one keeping track of the conversation by growing the list.  
> This is exactly what ChatGPT, Claude, and every chatbot does behind the scenes.

---

## Part 5 — Loading PDF Documents

Now we shift to the main topic of today: **working with documents**.

In a RAG (Retrieval-Augmented Generation) system, the first step is always:  
**Load your documents into a format the LLM can work with.**

LangChain's `PyPDFLoader` reads a PDF and converts each page into a **Document** object with:
- `.page_content` — the actual text
- `.metadata` — info about where the text came from (page number, source file, etc.)

### Setup: Place your PDFs
Put 2 PDF files in a `pdfs/` folder next to this notebook.  
Name them anything you like (e.g., `paper1.pdf`, `notes.pdf`).

In [39]:
# Load the first PDF
loader1 = PyPDFLoader(file_path="Gopalswamy_Doraiswamy_Naidu.pdf")
pages1 = loader1.load()

# Load the second PDF
loader2 = PyPDFLoader(file_path="A._P._J._Abdul_Kalam.pdf")
pages2 = loader2.load()

print(f"Loaded Gopalswamy_Doraiswamy_Naidu.pdf — {len(pages1)} pages")
print(f"Loaded A._P._J._Abdul_Kalam.pdf — {len(pages2)} pages")

Loaded Gopalswamy_Doraiswamy_Naidu.pdf — 4 pages
Loaded A._P._J._Abdul_Kalam.pdf — 30 pages


### What does a Document object look like?

In [40]:
# Look at the first page
first_page = pages1[0]

print("=== METADATA ===")
print(first_page.metadata)

print("\n=== PAGE CONTENT (first 500 chars) ===")
print(first_page.page_content[:500])

=== METADATA ===
{'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/145.0.0.0 Safari/537.36', 'creationdate': '2026-03-09T06:40:02+00:00', 'title': 'Gopalswamy Doraiswamy Naidu - Wikipedia', 'moddate': '2026-03-09T06:40:02+00:00', 'source': 'Gopalswamy_Doraiswamy_Naidu.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}

=== PAGE CONTENT (first 500 chars) ===
Gopalaswamy Doraiswamy Naidu
Gopalaswamy Doraiswamy Naidu
Born 23 March 1893
Kalangal, Coimbatore District,
Madras Presidency, British India
(now Tamil Nadu, India)
Died 4 January 1974 (aged 80)
Coimbatore, Tamil Nadu, India
CitizenshipIndian
Known for Technical redesigner, Industrialist,
businessman, photographer and
philanthropist
Scientific career
Fields Electrical, mechanics, automotive,
agriculture
Gopalswamy Doraiswamy Naidu
G. D. Naidu (Gopalaswamy Doraiswamy Naidu) (23
March 1893 – 4 Jan


> **Notice:** Each Document already comes with metadata like `source` (file path) and `page` (page number).  
> LangChain adds this automatically from the PDF. We can add our own metadata too — that's coming up next.

In [42]:
all_pages = pages1 + pages2
print(all_pages)

[Document(metadata={'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/145.0.0.0 Safari/537.36', 'creationdate': '2026-03-09T06:40:02+00:00', 'title': 'Gopalswamy Doraiswamy Naidu - Wikipedia', 'moddate': '2026-03-09T06:40:02+00:00', 'source': 'Gopalswamy_Doraiswamy_Naidu.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Gopalaswamy Doraiswamy Naidu\nGopalaswamy Doraiswamy Naidu\nBorn 23 March 1893\nKalangal, Coimbatore District,\nMadras Presidency, British India\n(now Tamil Nadu, India)\nDied 4 January 1974 (aged 80)\nCoimbatore, Tamil Nadu, India\nCitizenshipIndian\nKnown for Technical redesigner, Industrialist,\nbusinessman, photographer and\nphilanthropist\nScientific career\nFields Electrical, mechanics, automotive,\nagriculture\nGopalswamy Doraiswamy Naidu\nG. D. Naidu (Gopalaswamy Doraiswamy Naidu) (23\nMarch 1893 – 4 January 1974) was an Indian technical\nredesigner and industrial pio

---
## Part 6 — Chunking with RecursiveCharacterTextSplitter

### Why do we need to split documents?

A full page of text can be **thousands of characters**. When we later search for relevant info, we want **small, focused pieces** — not entire pages.

**`RecursiveCharacterTextSplitter`** splits text smartly:
1. First tries to split on `\n\n` (paragraph breaks)
2. Then `\n` (line breaks)
3. Then spaces
4. Then individual characters

It "recurses" through these separators to keep chunks as meaningful as possible.

### Two key parameters:
- **`chunk_size`** = max characters per chunk (we'll use 1000)
- **`chunk_overlap`** = how many characters overlap between consecutive chunks (we'll use 200)

**Why overlap?** So that a sentence split across two chunks still appears in full in at least one of them.

In [43]:
# Create the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Split all pages into chunks
chunks = text_splitter.split_documents(all_pages)

print(f"Total pages: {len(all_pages)}")
print(f"Total chunks after splitting: {len(chunks)}")
print(f"\nAverage chunks per page: {len(chunks) / len(all_pages):.1f}")

Total pages: 34
Total chunks after splitting: 149

Average chunks per page: 4.4


In [44]:
# Inspect a chunk
sample_chunk = chunks[0]

print("=== CHUNK METADATA ===")
print(sample_chunk.metadata)

print(f"\n=== CHUNK TEXT ({len(sample_chunk.page_content)} chars) ===")
print(sample_chunk.page_content[:300], "...")

=== CHUNK METADATA ===
{'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/145.0.0.0 Safari/537.36', 'creationdate': '2026-03-09T06:40:02+00:00', 'title': 'Gopalswamy Doraiswamy Naidu - Wikipedia', 'moddate': '2026-03-09T06:40:02+00:00', 'source': 'Gopalswamy_Doraiswamy_Naidu.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}

=== CHUNK TEXT (949 chars) ===
Gopalaswamy Doraiswamy Naidu
Gopalaswamy Doraiswamy Naidu
Born 23 March 1893
Kalangal, Coimbatore District,
Madras Presidency, British India
(now Tamil Nadu, India)
Died 4 January 1974 (aged 80)
Coimbatore, Tamil Nadu, India
CitizenshipIndian
Known for Technical redesigner, Industrialist,
businessma ...


> **Notice:** The metadata from the original page is **preserved** in each chunk.  
> So every chunk knows which file and page it came from. This is powerful.

In [46]:
# Let's see the overlap in action — compare the end of chunk 0 with the start of chunk 1
print("=== END of chunk 0 (last 200 chars) ===")
print(chunks[0].page_content[-200:])

print("\n=== START of chunk 1 (first 200 chars) ===")
print(chunks[1].page_content[:200])

=== END of chunk 0 (last 200 chars) ===
he manufacture of the first electric motor in India. His
contributions were primarily industrial but also
spanned the fields of electrical, mechanical,
agricultural (hybrid cultivation) and automobile

=== START of chunk 1 (first 200 chars) ===
contributions were primarily industrial but also
spanned the fields of electrical, mechanical,
agricultural (hybrid cultivation) and automobile
engineering.[2][3] Naidu developed an independently
inte


> **See the overlap?** The same text appears at the end of chunk 0 and the start of chunk 1.  
> This ensures no information is lost at chunk boundaries.

---
## Part 7 — Enriching Chunks with Custom Metadata

The default metadata (`source`, `page`) is useful, but we can add more.  
This is important because later, when you have hundreds of documents, you'll want to filter by things like:
- Which file did this come from? (`filename`)
- What type of document is it? (`source_type`)
- When was it uploaded? (`upload_date`)

Let's add custom metadata to every chunk.

In [49]:
# Define metadata for each PDF
pdf_metadata = {
    "Gopalswamy_Doraiswamy_Naidu.pdf": {"source_type": "paper"},
    "A._P._J._Abdul_Kalam.pdf": {"source_type": "notes"}
}

today = str(date.today())

# Enrich each chunk with custom metadata
for chunk in chunks:
    source_path = chunk.metadata.get("source", "")
    filename = os.path.basename(source_path)
    
    chunk.metadata["filename"] = filename
    chunk.metadata["page_number"] = chunk.metadata.get("page", 0)
    chunk.metadata["upload_date"] = today
    chunk.metadata["source_type"] = pdf_metadata.get(filename, {}).get("source_type", "unknown")

print("Metadata enrichment done!")
print(f"\nSample chunk metadata:")
print(chunks[0].metadata)

Metadata enrichment done!

Sample chunk metadata:
{'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/145.0.0.0 Safari/537.36', 'creationdate': '2026-03-09T06:40:02+00:00', 'title': 'Gopalswamy Doraiswamy Naidu - Wikipedia', 'moddate': '2026-03-09T06:40:02+00:00', 'source': 'Gopalswamy_Doraiswamy_Naidu.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'filename': 'Gopalswamy_Doraiswamy_Naidu.pdf', 'page_number': 0, 'upload_date': '2026-04-14', 'source_type': 'paper'}


In [50]:
# See all unique filenames and source types across chunks
filenames = set(c.metadata["filename"] for c in chunks)
source_types = set(c.metadata["source_type"] for c in chunks)
page_numbers = set(c.metadata["page_number"] for c in chunks)

print(f"Filenames:    {filenames}")
print(f"Source types: {source_types}")
print(f"Page numbers: {sorted(page_numbers)}")

Filenames:    {'A._P._J._Abdul_Kalam.pdf', 'Gopalswamy_Doraiswamy_Naidu.pdf'}
Source types: {'notes', 'paper'}
Page numbers: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]


---
## Part 8 — Filtering Chunks by Metadata

Now comes the payoff: **selective retrieval**.  
Instead of searching through ALL chunks, you can narrow down to only the ones that match certain criteria.

This is how real RAG systems work:
- "Only search in the uploaded textbook, not the lecture notes"
- "Only look at pages 5-10"
- "Only search documents uploaded today"

Let's build a simple filter function.

In [51]:
def filter_chunks(chunks, **filters):
    """
    Filter chunks by metadata fields.
    
    Usage:
        filter_chunks(chunks, filename="paper1.pdf")
        filter_chunks(chunks, source_type="notes", page_number=2)
    """
    results = []
    for chunk in chunks:
        match = True
        for key, value in filters.items():
            if chunk.metadata.get(key) != value:
                match = False
                break
        if match:
            results.append(chunk)
    return results

print("filter_chunks function defined!")

filter_chunks function defined!


### Filter Example 1 — Get chunks from one specific file

In [52]:
# Filter: only chunks from the first PDF
file1_chunks = filter_chunks(chunks, filename="Gopalswamy_Doraiswamy_Naidu.pdf")

print(f"Total chunks: {len(chunks)}")
print(f"Chunks from 'Gopalswamy_Doraiswamy_Naidu.pdf': {len(file1_chunks)}")
print(f"\nFirst matching chunk preview:")
print(file1_chunks[0].page_content[:200], "...")

Total chunks: 149
Chunks from 'Gopalswamy_Doraiswamy_Naidu.pdf': 13

First matching chunk preview:
Gopalaswamy Doraiswamy Naidu
Gopalaswamy Doraiswamy Naidu
Born 23 March 1893
Kalangal, Coimbatore District,
Madras Presidency, British India
(now Tamil Nadu, India)
Died 4 January 1974 (aged 80)
Coimb ...


### Filter Example 2 — Get chunks from a specific page

In [53]:
# Filter: only chunks from page 0 (first page) of the second PDF
page0_chunks = filter_chunks(chunks, filename="A._P._J._Abdul_Kalam.pdf", page_number=0)

print(f"Chunks from 'A._P._J._Abdul_Kalam.pdf', page 0: {len(page0_chunks)}")
for i, c in enumerate(page0_chunks):
    print(f"\n--- Chunk {i} ({len(c.page_content)} chars) ---")
    print(c.page_content[:150], "...")

Chunks from 'A._P._J._Abdul_Kalam.pdf', page 0: 3

--- Chunk 0 (953 chars) ---
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan  ...

--- Chunk 1 (965 chars) ---
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four de ...

--- Chunk 2 (585 chars) ---
Party and the then-opposition Indian National
Congress. He was widely referred to as the "People's
President". He engaged in teaching, writing and pub ...


### Filter Example 3 — Get chunks by source type

In [54]:
# Filter: only chunks tagged as "notes"
notes_chunks = filter_chunks(chunks, source_type="notes")

# Filter: only chunks tagged as "paper"
paper_chunks = filter_chunks(chunks, source_type="paper")

print(f"'notes' chunks:  {len(notes_chunks)}")
print(f"'paper' chunks:  {len(paper_chunks)}")
print(f"Total:           {len(notes_chunks) + len(paper_chunks)}")

'notes' chunks:  136
'paper' chunks:  13
Total:           149


---
## Recap

| Step | What we did | Why it matters |
|------|-------------|----------------|
| **ChatOllama** | Connected to Ollama through LangChain | Cleaner code, integrates with chains |
| **Chain (Prompt \| LLM \| Parser)** | Built a reusable query pipeline | The core pattern you'll use every day |
| **ConversationBufferMemory** | Gave the LLM memory of past messages | Makes chat feel natural |
| **PyPDFLoader** | Loaded PDFs as Document objects | First step in any RAG pipeline |
| **RecursiveCharacterTextSplitter** | Split pages into overlapping chunks | Small chunks = better retrieval |
| **Custom metadata** | Added filename, page, date, type | Enables selective filtering |
| **filter_chunks()** | Filtered chunks by metadata | Foundation of targeted retrieval |

### What's next (Day 3):
We'll take these chunks, **embed** them into vectors, store them in **ChromaDB**, and build a **hybrid search** RAG pipeline.

---